# Embedding Space Geometry

Analyze the geometric structure of token embeddings: isotropy,
nearest neighbors, embed-unembed alignment, and effective dimension.

## Why This Matters

Embedding space geometry examines the structure of the model's token representation spaces. The embedding and unembedding matrices define the model's interface with language, and their geometry constrains what semantic relationships the model can represent.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.embedding_space_geometry import (
    embedding_isotropy, embedding_nearest_neighbors,
    embed_unembed_alignment, embedding_effective_dimension,
    embedding_geometry_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print('Model ready')

## Isotropy

Are embeddings spread uniformly across dimensions or clustered?

In [ ]:
result = embedding_isotropy(model)
print(f"Mean cosine similarity: {result['mean_cosine']:.4f}")
print(f"Isotropy score: {result['isotropy_score']:.4f}")
print(f"Std of cosines: {result['std_cosine']:.4f}")
print(f"\nIsotropy 1.0 = perfectly uniform spread")
print(f"Lower values = more clustered embeddings")

## Nearest Neighbors

Which tokens have the most similar embeddings?

In [ ]:
result = embedding_nearest_neighbors(model, token_ids=[1, 10, 50, 99], k=5)
for entry in result['per_token']:
    print(f"\nToken {entry['token_id']}:")
    for n in entry['neighbors']:
        print(f"  -> token {n['token_id']}: cos={n['cosine']:.4f}")

## Embed-Unembed Alignment

How aligned are the embedding and unembedding vectors for each token?

In [ ]:
result = embed_unembed_alignment(model)
print(f"Mean alignment: {result['mean_alignment']:.4f}")
print(f"Std alignment: {result['std_alignment']:.4f}")
print(f"\nPer-token alignment (first 20):")
for i, a in enumerate(result['per_token_alignment'][:20]):
    bar = '█' * int(abs(float(a)) * 20)
    print(f"  Token {i:3d}: {float(a):+.4f} {bar}")

## Effective Dimension

How many dimensions does the embedding space effectively use?

In [ ]:
result = embedding_effective_dimension(model)
print(f"Effective rank: {result['effective_rank']:.2f} / {result['d_model']}")
print(f"Utilization: {result['utilization']:.3f}")
print(f"Top-5 singular values: {result['top_singular_values'][:5]}")

## Geometry Summary

In [ ]:
result = embedding_geometry_summary(model)
print(f"Isotropy: {result['isotropy_score']:.4f}")
print(f"Mean embed-unembed alignment: {result['mean_embed_unembed_alignment']:.4f}")
print(f"Effective rank: {result['effective_rank']:.2f} / {result['d_model']}")
print(f"Utilization: {result['utilization']:.3f}")